In [15]:
print("hej hej!")

hej hej!


In [42]:
import numpy as np
import plotly.graph_objects as go
import json
import os

# ----------------------------
# Inställningar
# ----------------------------
n_cylinders = 10
x_range = (-12, 12)
y_range = (-12, 12)
radius_range = (0.9, 2.0)
height = 40.0
spacing = 2.5
max_attempts = 2000

output_dir = "renders_gl"
os.makedirs(output_dir, exist_ok=True)

rng = np.random.default_rng()

def overlaps(x0, y0, r0, cylinders):
    for x1, y1, r1, _ in cylinders:
        if np.hypot(x0 - x1, y0 - y1) < (r0 + r1 + spacing):
            return True
    return False

# ----------------------------
# Generera cylindrar utan överlapp
# ----------------------------
cylinders = []

for _ in range(n_cylinders):
    placed = False
    for _attempt in range(max_attempts):
        r = rng.uniform(*radius_range)
        x0 = rng.uniform(x_range[0] + r, x_range[1] - r)
        y0 = rng.uniform(y_range[0] + r, y_range[1] - r)

        if not overlaps(x0, y0, r, cylinders):
            cylinders.append((x0, y0, r, height))
            placed = True
            break

    if not placed:
        print("Kunde inte placera alla cylindrar utan överlapp.")
        break

height_max = height

# ----------------------------
# Spara scen + cylindrar i JSON
# ----------------------------
scene_data = {
    "scene": {
        "x_range": list(x_range),
        "y_range": list(y_range),
        "height": float(height),
        "spacing": float(spacing)
    },
    "cylinders": []
}

for i, (x0, y0, r, h) in enumerate(cylinders):
    scene_data["cylinders"].append({
        "id": i,
        "x": float(x0),
        "y": float(y0),
        "radius": float(r),
        "height": float(h)
    })

json_path = os.path.join(output_dir, "scene.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(scene_data, f, indent=2)

print(f"Scen sparad i {json_path}")

# ----------------------------
# Visualisering (Plotly)
# ----------------------------
colors = [
    "red", "blue", "green", "orange", "purple",
    "cyan", "magenta", "yellow", "brown", "pink"
]

fig = go.Figure()
theta = np.linspace(0, 2 * np.pi, 50)

for i, (x0, y0, r, h) in enumerate(cylinders):
    z = np.linspace(0, h, 30)
    theta_grid, z_grid = np.meshgrid(theta, z)

    x = x0 + r * np.cos(theta_grid)
    y = y0 + r * np.sin(theta_grid)

    color = colors[i % len(colors)]

    fig.add_trace(go.Surface(
        x=x,
        y=y,
        z=z_grid,
        surfacecolor=np.zeros_like(z_grid),
        colorscale=[[0, color], [1, color]],
        showscale=False,
        opacity=0.9
    ))

fig.update_layout(
    title="Parallella cylindrar (samma höjd)",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
    width=900,
    height=700
)

fig.show()

Scen sparad i renders_gl/scene.json


In [43]:
import numpy as np
import moderngl
from PIL import Image, ImageOps
import os
import json
import glob

# ----------------------------
# Läs in scenen från JSON
# ----------------------------
output_dir = "renders_gl"
json_path = os.path.join(output_dir, "scene.json")

with open(json_path, "r", encoding="utf-8") as f:
    scene_data = json.load(f)

cylinders = [
    (c["x"], c["y"], c["radius"], c["height"])
    for c in scene_data["cylinders"]
]

x_range = tuple(scene_data["scene"]["x_range"])
y_range = tuple(scene_data["scene"]["y_range"])
height = float(scene_data["scene"]["height"])

# ----------------------------
# Hämta bakgrundsbilder
# ----------------------------
bg_paths = []
bg_paths.extend(glob.glob("backgrounds/*.jpg"))
bg_paths.extend(glob.glob("backgrounds/*.jpeg"))
bg_paths.extend(glob.glob("backgrounds/*.png"))

if not bg_paths:
    raise FileNotFoundError("Hittade inga bakgrundsbilder i mappen 'backgrounds'.")

# ----------------------------
# Hjälpfunktioner
# ----------------------------
def perspective(fovy_deg, aspect, near, far):
    f = 1.0 / np.tan(np.radians(fovy_deg) / 2.0)
    m = np.zeros((4, 4), dtype=np.float32)
    m[0, 0] = f / aspect
    m[1, 1] = f
    m[2, 2] = (far + near) / (near - far)
    m[2, 3] = (2 * far * near) / (near - far)
    m[3, 2] = -1.0
    return m

def look_at(eye, target, up):
    eye = np.asarray(eye, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    up = np.asarray(up, dtype=np.float32)

    f = target - eye
    f = f / np.linalg.norm(f)

    s = np.cross(f, up)
    s = s / np.linalg.norm(s)

    u = np.cross(s, f)

    m = np.eye(4, dtype=np.float32)
    m[0, :3] = s
    m[1, :3] = u
    m[2, :3] = -f
    m[0, 3] = -np.dot(s, eye)
    m[1, 3] = -np.dot(u, eye)
    m[2, 3] = np.dot(f, eye)
    return m

def camera_position(radius, elev_deg, azim_deg):
    elev = np.radians(elev_deg)
    azim = np.radians(azim_deg)
    x = radius * np.cos(elev) * np.cos(azim)
    y = radius * np.cos(elev) * np.sin(azim)
    z = radius * np.sin(elev)
    return np.array([x, y, z], dtype=np.float32)

def cylinder_mesh(segments=40):
    theta = np.linspace(0, 2 * np.pi, segments, endpoint=False)
    vertices = []
    indices = []

    for t in theta:
        x = np.cos(t)
        y = np.sin(t)
        vertices.append((x, y, 0.0))
        vertices.append((x, y, 1.0))

    for i in range(segments):
        i0 = 2 * i
        i1 = 2 * i + 1
        i2 = 2 * ((i + 1) % segments)
        i3 = i2 + 1
        indices.extend([i0, i2, i1])
        indices.extend([i1, i2, i3])

    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.int32)

def load_and_crop_background(path, size, rng):
    img = Image.open(path).convert("RGB")

    target_w, target_h = size
    target_aspect = target_w / target_h

    w, h = img.size
    img_aspect = w / h

    # Bestäm crop-storlek
    if img_aspect > target_aspect:
        # Bilden är för bred → crop i bredd
        new_h = h
        new_w = int(h * target_aspect)
    else:
        # Bilden är för hög → crop i höjd
        new_w = w
        new_h = int(w / target_aspect)

    # Slumpa crop-position
    max_x = w - new_w
    max_y = h - new_h

    x0 = int(rng.uniform(0, max_x)) if max_x > 0 else 0
    y0 = int(rng.uniform(0, max_y)) if max_y > 0 else 0

    crop = img.crop((x0, y0, x0 + new_w, y0 + new_h))

    # Resize till outputstorlek
    crop = crop.resize((target_w, target_h), Image.Resampling.LANCZOS)

    return crop

# ----------------------------
# Inställningar
# ----------------------------
n_images = 20
image_dir = os.path.join(output_dir, "images")
os.makedirs(image_dir, exist_ok=True)

W, H = 512, 512
rng = np.random.default_rng()

ctx = moderngl.create_standalone_context()
ctx.enable(moderngl.DEPTH_TEST)

color_tex = ctx.texture((W, H), components=4)
depth_rb = ctx.depth_renderbuffer((W, H))
fbo = ctx.framebuffer(color_attachments=[color_tex], depth_attachment=depth_rb)

# ----------------------------
# Shader för cylindrar
# ----------------------------
cyl_prog = ctx.program(
    vertex_shader="""
    #version 330
    in vec3 in_pos;
    uniform mat4 M;
    uniform mat4 V;
    uniform mat4 P;
    void main() {
        gl_Position = P * V * M * vec4(in_pos, 1.0);
    }
    """,
    fragment_shader="""
    #version 330
    uniform vec3 color;
    out vec4 f_color;
    void main() {
        f_color = vec4(color, 1.0);
    }
    """
)

# ----------------------------
# Shader + geometri för bakgrund
# ----------------------------
bg_prog = ctx.program(
    vertex_shader="""
    #version 330
    in vec2 in_pos;
    in vec2 in_uv;
    out vec2 v_uv;
    void main() {
        v_uv = in_uv;
        gl_Position = vec4(in_pos, 0.0, 1.0);
    }
    """,
    fragment_shader="""
    #version 330
    uniform sampler2D bg_tex;
    in vec2 v_uv;
    out vec4 f_color;
    void main() {
        f_color = texture(bg_tex, v_uv);
    }
    """
)

bg_quad = np.array([
    -1.0, -1.0, 0.0, 0.0,
     1.0, -1.0, 1.0, 0.0,
    -1.0,  1.0, 0.0, 1.0,
     1.0,  1.0, 1.0, 1.0,
], dtype=np.float32)

bg_vbo = ctx.buffer(bg_quad.tobytes())
bg_vao = ctx.vertex_array(bg_prog, [(bg_vbo, "2f 2f", "in_pos", "in_uv")])

# ----------------------------
# Enhets-cylinder på GPU
# ----------------------------
vertices, indices = cylinder_mesh(segments=40)
vbo = ctx.buffer(vertices.tobytes())
ibo = ctx.buffer(indices.tobytes())
vao = ctx.vertex_array(cyl_prog, [(vbo, "3f", "in_pos")], ibo)

# ----------------------------
# Färger
# ----------------------------
colors = np.array([
    [1.0, 0.2, 0.2],
    [0.2, 0.4, 1.0],
    [0.2, 0.8, 0.3],
    [1.0, 0.6, 0.1],
    [0.6, 0.2, 0.8],
    [0.1, 0.9, 0.9],
    [1.0, 0.2, 0.8],
    [0.9, 0.9, 0.2],
    [0.5, 0.3, 0.1],
    [1.0, 0.5, 0.7],
], dtype=np.float32)

# ----------------------------
# Kamera-inställningar
# ----------------------------
height_max = max(h for *_xyz, h in cylinders) if cylinders else 1.0
scene_center = np.array([0.0, 0.0, height_max * 0.5], dtype=np.float32)

scene_width = x_range[1] - x_range[0]
scene_depth = y_range[1] - y_range[0]
scene_diagonal = np.sqrt(scene_width**2 + scene_depth**2 + height_max**2)

# Justera dessa om kameran ska närmare/längre bort
min_cam_radius = 1.0 * scene_diagonal
max_cam_radius = 1.5 * scene_diagonal

camera_data = []

# ----------------------------
# Render-loop
# ----------------------------
for i in range(n_images):
    elev = float(rng.uniform(15, 55))
    azim = float(rng.uniform(0, 360))
    cam_radius = float(rng.uniform(min_cam_radius, max_cam_radius))

    eye = camera_position(cam_radius, elev, azim)
    target = scene_center
    up = np.array([0.0, 0.0, 1.0], dtype=np.float32)

    V = look_at(eye, target, up)
    P = perspective(35.0, W / H, 0.1, 100.0)

    # Välj och förbered bakgrund
    bg_path = rng.choice(bg_paths)
    bg_img = load_and_crop_background(bg_path, (W, H), rng)    
    bg_arr = np.array(bg_img, dtype=np.uint8)
    bg_tex = ctx.texture((W, H), 3, data=bg_arr.tobytes())
    bg_tex.filter = (moderngl.LINEAR, moderngl.LINEAR)

    # Rendera bakgrunden
    fbo.use()
    ctx.clear(0.0, 0.0, 0.0, 1.0, depth=1.0)

    ctx.disable(moderngl.DEPTH_TEST)
    bg_tex.use(location=0)
    bg_prog["bg_tex"].value = 0
    bg_vao.render(moderngl.TRIANGLE_STRIP)

    # Rendera cylindrar ovanpå
    ctx.enable(moderngl.DEPTH_TEST)

    cyl_prog["V"].write(V.T.tobytes())
    cyl_prog["P"].write(P.T.tobytes())

    for j, (x0, y0, r, h) in enumerate(cylinders):
        M = np.array([
            [r,   0.0, 0.0, x0],
            [0.0, r,   0.0, y0],
            [0.0, 0.0, h,   0.0],
            [0.0, 0.0, 0.0, 1.0]
        ], dtype=np.float32)

        cyl_prog["M"].write(M.T.tobytes())
        cyl_prog["color"].value = tuple(colors[j % len(colors)])
        vao.render()

    data = fbo.read(components=4, alignment=1)
    img = Image.frombytes("RGBA", (W, H), data).transpose(Image.FLIP_TOP_BOTTOM)

    filename = f"frame_{i:03d}.png"
    img.save(os.path.join(image_dir, filename))

    camera_data.append({
        "image": filename,
        "camera_matrix": V.tolist(),
        "background": os.path.basename(bg_path)
    })

    bg_tex.release()

with open(os.path.join(output_dir, "camera_matrices.json"), "w", encoding="utf-8") as f:
    json.dump(camera_data, f, indent=2)

print(f"Klart: {n_images} bilder sparade i '{image_dir}'")
print(f"Kameramatrisen finns i '{os.path.join(output_dir, 'camera_matrices.json')}'")

/home2/johannae/anaconda3/envs/data_gen/lib/python3.11/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (99996755 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Klart: 20 bilder sparade i 'renders_gl/images'
Kameramatrisen finns i 'renders_gl/camera_matrices.json'
